# 01 - Data acquisition and provenance

This is the starting point for the Resume Repository (RR) data pipeline. Run the notebook from top to bottom. It creates the required local folders, safely downloads missing automatic data when requested, verifies every registered file, and tells you exactly what remains to be done.

The study period is **1 January 2015 through 31 December 2020**. Complete third-party datasets remain local and outside Git. The population input is the original annual Queensland statewide series used by TCN_starNLL. It changes on 1 April and is never routed into the regional histories.

### What you need to do

1. In the settings cell below, leave `DOWNLOAD_MISSING_DATA = False` for a status check or change it to `True` for the first automatic download.
2. Run every cell in order.
3. Follow the final status table for any manual source files still required.
4. Continue to Notebook 02 only when the final gate reports **PASS**.


## 1. Choose whether to download missing automatic data

For ordinary use, edit only `DOWNLOAD_MISSING_DATA` in the next cell. `False` checks existing files without using the internet. `True` obtains only the missing sources declared in the frozen 2020 registry and never overwrites an existing canonical artifact. There is deliberately no snapshot or force mode because either could introduce a later source vintage.

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

candidate = Path.cwd().resolve()
PROJECT_ROOT = next(
    path for path in (candidate, *candidate.parents)
    if (path / 'environment.yml').is_file() and (path / 'data').is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_provenance import (
    audit_manifest,
    duplicate_and_gap_audit,
    load_yaml,
    source_inventory,
)

CONFIG = load_yaml(PROJECT_ROOT / 'config' / 'data_acquisition.yml')
REGISTRY = load_yaml(PROJECT_ROOT / CONFIG['paths']['source_registry_path'])
CONTRACTS = load_yaml(PROJECT_ROOT / 'data' / 'schemas' / 'source_contracts.yml')
MANIFEST_PATH = PROJECT_ROOT / CONFIG['paths']['manifest_path']

print(f'RR root: {PROJECT_ROOT}')
print(f'Python:  {sys.executable}')
print(f'Study:   {CONFIG["study"]["study_start"]} to {CONFIG["study"]["study_end"]}')

In [ ]:
# -------- USER SETTINGS --------
DOWNLOAD_MISSING_DATA = False
# -------------------------------

mode = 'missing' if DOWNLOAD_MISSING_DATA else 'verify'
network_allowed = mode != 'verify'
run_environment = os.environ.copy()
run_environment[CONFIG['controls']['mode_environment_variable']] = mode
run_environment[CONFIG['controls']['network_environment_variable']] = (
    '1' if network_allowed else '0'
)

display(pd.DataFrame([{
    'acquisition_mode': mode,
    'network_allowed': network_allowed,
    'existing_files_overwritten': False,
    'action': (
        'Download only missing automatic files' if mode == 'missing'
        else 'Check local files only'
    ),
}]))

## 2. Review the public data sources

AEMO and Open-Meteo are requested only for 2015–2020. ABS Regional Population 2018–19, ASGS 2016, and the CER 2020 snapshot are exact pinned downloads whose byte size and SHA-256 must match before admission. Queensland calendar records are exact pinned manual artifacts. No current rolling provider file is accepted and no later file is downloaded and trimmed.

In [ ]:
source_table = pd.DataFrame(REGISTRY['sources'])[
    ['source_id', 'provider', 'product', 'acquisition', 'coverage_start', 'coverage_end', 'landing_page']
]
display(source_table)

## 3. Create the local data folders and inspect current status

The required folders are created automatically. Raw provider files and generated datasets are ignored by Git. Reviewed manifests, schemas, and small lawful or synthetic fixtures are the only data-layer artifacts intended for version control.

In [ ]:
for relative in [
    'data/raw/aemo',
    'data/raw/aemo/p5min_qld_monthly',
    'data/raw/open_meteo',
    'data/external/abs/asgs',
    'data/external/cer',
    'data/external/qld_calendar/public_holidays',
    'data/external/qld_calendar/school_terms',
    'data/interim',
    'data/processed',
    'data/manifests',
    'data/schemas',
    'data/fixtures',
]:
    (PROJECT_ROOT / relative).mkdir(parents=True, exist_ok=True)

display(source_inventory(PROJECT_ROOT, REGISTRY, MANIFEST_PATH))

## 4. Download or verify automatic data

This cell follows the setting selected above. It either checks local files or obtains only the missing sources in the frozen registry. Every dynamic request is bounded to 2015–2020; every versioned source is fixed at an edition released no later than 2020-12-31. AEMO P5MIN acquisition preserves the monthly QLD1 origin vintages and retains only 5–30 minute targets whose origins and target timestamps both remain inside 2015–2020. Population acquisition is also vintage-specific: 26 quarterly statewide workbooks and six annual regional workbooks are individually pinned by URL, byte size, and SHA-256. The implementation is kept in [`tools/acquire_rr_data.py`](../tools/acquire_rr_data.py).


In [ ]:
command = [
    sys.executable,
    str(PROJECT_ROOT / 'tools' / 'acquire_rr_data.py'),
    '--mode',
    mode,
]
print(f'Running automatic-data action: {mode}')
subprocess.run(
    command,
    cwd=PROJECT_ROOT,
    env=run_environment,
    check=True,
)

## 5. Verify file identity

Every registered file has a recorded size and SHA-256 checksum. The audit below recomputes both values. A changed or missing file fails immediately, even if its filename has not changed.

In [ ]:
manifest_audit = audit_manifest(PROJECT_ROOT, MANIFEST_PATH)
if manifest_audit.empty:
    display(Markdown('No manifest is displayed until the first local acquisition has been fingerprinted.'))
else:
    display(manifest_audit)
    if not manifest_audit[['exists', 'size_matches', 'sha256_matches', 'coverage_within_2020']].all().all():
        raise ValueError('At least one registered artifact failed its identity audit.')

## 6. Check time coverage and source keys

These checks confirm the AEMO dispatch timeline, the origin-vintage P5MIN six-horizon benchmark, and each regional hourly weather timeline before any joining or feature engineering. Missing timestamps are reported rather than silently filled. `DISPATCHREGIONSUM.DEMANDFORECAST` is never substituted for `P5MIN_REGIONSOLUTION.TOTALDEMAND`, and neither forecast field is a learned-model input.

In [ ]:
aemo_path = PROJECT_ROOT / CONFIG['aemo_dispatchregionsum']['canonical_output']
p5min_path = PROJECT_ROOT / CONFIG['aemo_p5min_regionsolution']['processed_output']
weather_path = PROJECT_ROOT / CONFIG['open_meteo']['canonical_output']

if aemo_path.is_file():
    aemo = pd.read_parquet(aemo_path)
    required = set(CONTRACTS['contracts']['aemo_dispatchregionsum']['required_columns'])
    missing = sorted(required.difference(aemo.columns))
    if missing:
        raise KeyError(f'AEMO artifact is missing fields: {missing}')
    if not aemo['REGIONID'].eq('QLD1').all():
        raise ValueError('AEMO artifact contains a region other than QLD1.')
    display(duplicate_and_gap_audit(
        aemo, timestamp_column='SETTLEMENTDATE', frequency='5min'
    ))

if p5min_path.is_file():
    p5min = pd.read_parquet(p5min_path)
    required = set(CONTRACTS['contracts']['aemo_p5min_regionsolution']['required_columns'])
    missing = sorted(required.difference(p5min.columns))
    if missing:
        raise KeyError(f'AEMO P5MIN artifact is missing fields: {missing}')
    if set(p5min['horizon_minutes'].astype(int).unique()) != {5, 10, 15, 20, 25, 30}:
        raise ValueError('AEMO P5MIN artifact does not contain exactly the six RR horizons.')
    if p5min.duplicated(['forecast_origin', 'target_timestamp', 'region_id']).any():
        raise ValueError('AEMO P5MIN artifact contains duplicate origin-target keys.')
    display(p5min.groupby('horizon_minutes').agg(first_origin=('forecast_origin', 'min'), last_origin=('forecast_origin', 'max'), rows=('forecast_origin', 'size')))

if weather_path.is_file():
    weather = pd.read_parquet(weather_path)
    required = set(CONTRACTS['contracts']['open_meteo_hourly']['required_columns'])
    missing = sorted(required.difference(weather.columns))
    if missing:
        raise KeyError(f'Weather artifact is missing fields: {missing}')
    if weather['region'].nunique() != 5:
        raise ValueError('Weather artifact does not contain exactly five regions.')
    display(duplicate_and_gap_audit(
        weather, timestamp_column='datetime', frequency='1h', group_columns=['region']
    ))

## 7. Register the exact Queensland calendar records

The only manual inputs are the exact Queensland calendar records listed—with fixed filenames, sizes, and checksums—in the source registry. Registration does not accept arbitrary files or command-line coverage dates.

After placing the declared calendar files in their exact paths, register them with:

```powershell
python tools/register_manual_source.py `
  --source-id qld_public_holidays `
  --notes "Exact frozen Queensland records through 2020"
```

Repeat with `qld_school_calendar`. Coverage and source-release dates come from the locked registry and cannot be widened at the command line.

In [ ]:
final_inventory = source_inventory(PROJECT_ROOT, REGISTRY, MANIFEST_PATH)
registry_by_id = {source['source_id']: source for source in REGISTRY['sources']}

def next_action(row):
    source = registry_by_id[row['source_id']]
    if row['status'] == 'registered and verified':
        return 'Ready — no action required'
    if source['acquisition'] in {'automatic_api', 'pinned_download'}:
        if row['status'] == 'not present locally':
            return 'Set DOWNLOAD_MISSING_DATA = True and rerun the notebook'
        return 'Stop: inspect this unregistered automatic artifact before continuing'
    folder = str(Path(source['artifacts'][0]['relative_path']).parent).replace('\\', '/') + '/'
    if row['status'] == 'not present locally':
        return f'Download from the source page and place the file in {folder}'
    return 'Register the present file using the command in Section 7'

final_inventory['next_action'] = final_inventory.apply(next_action, axis=1)
final_inventory['source_page'] = final_inventory['source_id'].map(
    lambda source_id: registry_by_id[source_id]['landing_page']
)
display(final_inventory[[
    'source_id', 'provider', 'status', 'next_action', 'source_page'
]])
all_sources_ready = final_inventory['status'].eq('registered and verified').all()
gate_message = (
    'ALL REQUIRED RR SOURCE ARTIFACTS ASSEMBLED: PASS'
    if all_sources_ready
    else 'ALL REQUIRED RR SOURCE ARTIFACTS ASSEMBLED: INCOMPLETE'
)
display(Markdown(f'### {gate_message}'))

## 8. Continue only after the acquisition gate passes

When the gate reports **PASS**, proceed to Notebook 02 for validation, unit harmonisation, causal availability joins, and preparation of the public modelling tables. If it reports **INCOMPLETE**, follow the `next_action` entry for each unfinished source and rerun this notebook. Notebook 02 does not redownload or silently repair source data.

## 9. TCN_starNLL source completion

The final public model additionally uses two causally available SOI channels
and regional installed distributed-solar capacity. Monthly SOI is acquired
from the Bureau of Meteorology and admitted at the first day of month M+2.
Regional capacity is reconstructed from the pinned Clean Energy Regulator
workbooks and the declared postcode-to-region contract. The acquisition tool
writes only the admitted through-2020 subset; it does not download a current
rolling product and trim it after admission.


In [ ]:
from src.tcn_star_nll_sources import build_publication_safe_soi, write_regional_solar_capacity

capacity_path = write_regional_solar_capacity(PROJECT_ROOT)
soi_origins = pd.date_range('2015-01-01', '2020-12-31 23:55', freq='5min')
soi_features = build_publication_safe_soi(
    soi_origins,
    PROJECT_ROOT / 'data/raw/bom/soi_monthly_through_2020.csv',
)
tcn_star_sources = {
    'regional_capacity_path': capacity_path.relative_to(PROJECT_ROOT).as_posix(),
    'soi_rows': len(soi_features),
    'soi_missing_values': int(soi_features.isna().sum().sum()),
    'latest_origin': soi_features.index.max(),
}
tcn_star_sources
